In [11]:
import copy
import os
import torch
import torch.nn as nn
from torch.optim import lr_scheduler
from torchvision import datasets, transforms, models

In [12]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

In [13]:
data_dir = "hymenoptera_data"
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
image_loader = {x: torch.utils.data.dataloader.DataLoader(
    image_datasets[x],
    batch_size=4,
    shuffle=True
) for x in ['train', 'val']}

In [19]:
resnet_18 = models.resnet18(pretrained=True)

num_features = resnet_18.fc.in_features

resnet_18.fc = nn.Linear(num_features, 2)

for i, child in enumerate(resnet_18.children()):
    print(f"Element {i}: {child}")

for i, module in enumerate(resnet_18.modules()):
    print(f"Module {i}: {module}")


Element 0: Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
Element 1: BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
Element 2: ReLU(inplace=True)
Element 3: MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
Element 4: Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kern

In [15]:
CUDA = torch.cuda.is_available()

if CUDA:
    resnet_18 = resnet_18.cuda()

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(resnet_18.fc.parameters(), lr=0.01, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer=optimizer, step_size=7, gamma=0.1)

In [16]:
epochs = 25

best_model_wts = copy.deepcopy(resnet_18.state_dict())
best_acc = 0.0

for epoch in range(epochs):

    for phase in ['train', 'val']:
        correct = 0
        total_loss = 0

        if phase == 'train':
            resnet_18.train()
        else:
            resnet_18.eval()

        for (inputs, labels) in image_loader[phase]:
            with torch.set_grad_enabled(phase == 'train'):
                if CUDA:
                    inputs = inputs.cuda()
                    labels = labels.cuda()

                if phase == 'train':
                    optimizer.zero_grad()

                outputs = resnet_18(inputs)
                (_, predicates) = torch.max(outputs, 1)
                correct += (predicates == labels).sum()

                loss = loss_func(outputs, labels)
                total_loss += loss

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

        epoch_loss = total_loss / len(image_datasets[phase])
        epoch_acc = correct.double() / len(image_datasets[phase])

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(resnet_18.state_dict())

    exp_lr_scheduler.step()

resnet_18.load_state_dict(best_model_wts)

train Loss: 0.2908 Acc: 0.6926
val Loss: 0.1940 Acc: 0.9281
train Loss: 0.4688 Acc: 0.7623
val Loss: 0.3055 Acc: 0.8562
train Loss: 0.8181 Acc: 0.6967
val Loss: 1.4267 Acc: 0.6340
train Loss: 1.5534 Acc: 0.7172
val Loss: 0.2750 Acc: 0.9412
train Loss: 0.5855 Acc: 0.8074
val Loss: 0.2861 Acc: 0.9346
train Loss: 1.0999 Acc: 0.7418
val Loss: 0.2486 Acc: 0.9412
train Loss: 0.5007 Acc: 0.8361
val Loss: 0.3378 Acc: 0.9281
train Loss: 0.4821 Acc: 0.8279
val Loss: 0.2872 Acc: 0.9346
train Loss: 0.2736 Acc: 0.8852
val Loss: 0.3046 Acc: 0.9150
train Loss: 0.2671 Acc: 0.8770
val Loss: 0.4002 Acc: 0.8954
train Loss: 0.2692 Acc: 0.8730
val Loss: 0.2921 Acc: 0.9216
train Loss: 0.3605 Acc: 0.8607
val Loss: 0.2528 Acc: 0.9346
train Loss: 0.3843 Acc: 0.8402
val Loss: 0.3295 Acc: 0.9020
train Loss: 0.2927 Acc: 0.8730
val Loss: 0.2579 Acc: 0.9216
train Loss: 0.3473 Acc: 0.8402
val Loss: 0.2981 Acc: 0.9216
train Loss: 0.3316 Acc: 0.8566
val Loss: 0.2770 Acc: 0.9216
train Loss: 0.2816 Acc: 0.8689
val Loss:

<All keys matched successfully>